In [12]:
import pandas as pd
df = pd.read_csv('../data/processed_v2/repositories_01.csv')
df.head()

,owner,name,stars,forks,watchers,languageCount,description,primaryLanguage,createdAt,pushedAt,languages,topics,isFork,isArchived
0,freeCodeCamp,freeCodeCamp,426893,41377,8588,6,freeCodeCamp.org's open-source codebase and cu...,TypeScript,2014-12-24T17:49:19Z,2025-08-31T09:33:14Z,"['TypeScript', 'JavaScript', 'CSS', 'Dockerfil...","['learn-to-code', 'nonprofits', 'programming',...",False,False
1,codecrafters-io,build-your-own-x,415801,38976,6247,1,Master programming by recreating your favorite...,Markdown,2018-05-09T12:03:18Z,2025-08-29T00:08:20Z,['Markdown'],"['programming', 'tutorials', 'tutorial-code', ...",False,False
2,sindresorhus,awesome,396445,31414,8011,0,😎 Awesome lists about all kinds of interesting...,NaN,2014-07-11T13:42:37Z,2025-07-18T18:37:33Z,[],"['awesome', 'awesome-list', 'unicorns', 'lists...",False,False
3,EbookFoundation,free-programming-books,366964,64025,9879,2,:books: Freely available programming books,Python,2013-10-11T06:50:37Z,2025-08-25T21:06:44Z,"['Python', 'HTML']","['education', 'books', 'list', 'resource', 'ha...",False,False
4,public-apis,public-apis,363505,38166,4360,2,A collective list of free APIs,Python,2016-03-20T23:49:42Z,2025-05-20T15:56:34Z,"['Python', 'Shell']","['api', 'public-apis', 'free', 'apis', 'list',...",False,False


In [13]:
df.shape

(100000, 14)

In [14]:
df['isFork'].value_counts()

isFork
False    100000
Name: count, dtype: int64

In [15]:
df['isArchived'].value_counts()

isArchived
False    92459
True      7541
Name: count, dtype: int64

In [17]:
df['createdAt'].head()

0    2014-12-24T17:49:19Z
1    2018-05-09T12:03:18Z
2    2014-07-11T13:42:37Z
3    2013-10-11T06:50:37Z
4    2016-03-20T23:49:42Z
Name: createdAt, dtype: str

In [18]:
df['pushedAt'] = pd.to_datetime(df['pushedAt'], utc=True)

In [19]:
df['createdAt'] = pd.to_datetime(df['createdAt'], utc=True)

In [20]:
df['pushedAt'].min()

Timestamp('2010-01-23 23:20:32+0000', tz='UTC')

In [21]:
df['pushedAt'].max()

Timestamp('2025-09-01 09:39:12+0000', tz='UTC')

In [23]:
df['createdAt'] = pd.to_datetime(df['createdAt'], utc=True)

In [24]:
df[['createdAt', 'pushedAt']].dtypes

createdAt    datetime64[us, UTC]
pushedAt     datetime64[us, UTC]
dtype: object

In [45]:
(df['repo_age_days'] < 0).sum()

np.int64(0)

In [ ]:
def validate_numbers(repository):
    stars = repository['stars']
    forks = repository['forks']
    watchers = repository['watchers']
    languageCount = repository['languageCount']

    if (stars < 0 or forks < 0 or watchers < 0 or languageCount < 0):
        return False
    return True

In [ ]:
from datetime import date
def validate_date(repository):
    createdAt = repository['createdAt']
    pushedAt = repository['pushedAt']

    if not isinstance(createdAt,date):
        return False
    if not isinstance(pushedAt,date):
        return False
    if repository['repo_age_days'] < 0:
        return False
    return True

In [ ]:
def validate_name(repository):
    name = repository['name']
    if pd.isna(name):
        return False
    if not isinstance(name,str):
        return False
    if name.strip() == "":
        return False
    return True

In [29]:
df.info()

<class 'pandas.DataFrame'>
Index: 99982 entries, 0 to 99999
Data columns (total 15 columns):
 #   Column           Non-Null Count  Dtype              
---  ------           --------------  -----              
 0   owner            99982 non-null  str                
 1   name             99980 non-null  str                
 2   stars            99982 non-null  int64              
 3   forks            99982 non-null  int64              
 4   watchers         99982 non-null  int64              
 5   languageCount    99982 non-null  int64              
 6   description      97192 non-null  str                
 7   primaryLanguage  91763 non-null  str                
 8   createdAt        99982 non-null  datetime64[us, UTC]
 9   pushedAt         99982 non-null  datetime64[us, UTC]
 10  languages        99982 non-null  str                
 11  topics           99982 non-null  str                
 12  isFork           99982 non-null  bool               
 13  isArchived       99982 non-null 

In [53]:
df.isna().any()

owner              False
name               False
stars              False
forks              False
watchers           False
languageCount      False
description         True
primaryLanguage     True
createdAt          False
pushedAt           False
languages          False
topics             False
isFork             False
isArchived         False
repo_age_days      False
dtype: bool

In [ ]:
#Function 1
import pandas as pd
def load_data(file_path):
    return pd.read_csv(file_path)

In [ ]:
#Function 2
def convert_dates(df):
    df['pushedAt'] = pd.to_datetime(df['pushedAt'], utc=True)
    df['createdAt'] = pd.to_datetime(df['createdAt'], utc=True)
    return df

In [51]:
def remove_empty_names(df):
    df = df[df['name'].fillna('').str.strip() != '']
    return df

In [ ]:
def remove_duplicates(df):
    df = df.drop_duplicates(subset=['owner', 'name'])
    return df

In [55]:
def calculate_repo_age(df):
    df['repo_age_days'] = (
        df['pushedAt'] - df['createdAt']
    ).dt.days
    return df

In [54]:
def remove_invalid_repo_age(df):
    df = df[df['repo_age_days'] >= 0]
    return df

In [61]:
import ast

def combine_text(df):
    def combine_row(row):
        description = row['description'] if pd.notna(row['description']) else ''
        primary_language = row['primaryLanguage'] if pd.notna(row['primaryLanguage']) else ''

        languages = ast.literal_eval(row['languages']) if row['languages'] else []
        topics = ast.literal_eval(row['topics']) if row['topics'] else []

        return ' '.join([
            str(row['name']),
            str(description),
            str(primary_language),
            *languages,
            *topics
        ])

    df['combined_text'] = df.apply(combine_row, axis=1)

    return df


In [60]:
df.info()

<class 'pandas.DataFrame'>
Index: 99949 entries, 0 to 99999
Data columns (total 15 columns):
 #   Column           Non-Null Count  Dtype              
---  ------           --------------  -----              
 0   owner            99949 non-null  str                
 1   name             99949 non-null  str                
 2   stars            99949 non-null  int64              
 3   forks            99949 non-null  int64              
 4   watchers         99949 non-null  int64              
 5   languageCount    99949 non-null  int64              
 6   description      97160 non-null  str                
 7   primaryLanguage  91733 non-null  str                
 8   createdAt        99949 non-null  datetime64[us, UTC]
 9   pushedAt         99949 non-null  datetime64[us, UTC]
 10  languages        99949 non-null  str                
 11  topics           99949 non-null  str                
 12  isFork           99949 non-null  bool               
 13  isArchived       99949 non-null 